In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import curve_fit
import matplotlib.pyplot as plt

# Dados da sua tabela (exemplo com alguns pontos para ilustração)
# Certifique-se de usar TODOS os seus pontos médios reais da tabela
data_l0_performance = {
    'Tam_L0': np.array([10, 20, 30, 40, 50, 60, 70, 80, 90, 100,
                       200, 300, 400, 500, 600, 700, 800, 900, 1000,
                       2000, 3000, 4000, 5000, 6000, 7000, 8000, 9000, 10000,
                       20000, 30000, 40000, 50000, 60000, 70000, 80000, 90000, 100000,
                       110000, 120000, 130000, 140000, 150000, 160000, 170000, 180000, 190000, 200000]),
    'Acuracia_Media': np.array([0.067, 0.105, 0.140, 0.160, 0.178, 0.193, 0.207, 0.224, 0.235, 0.247,
                               0.333, 0.391, 0.435, 0.468, 0.491, 0.513, 0.529, 0.546, 0.559,
                               0.640, 0.679, 0.705, 0.721, 0.736, 0.747, 0.755, 0.763, 0.769,
                               0.806, 0.826, 0.838, 0.847, 0.853, 0.859, 0.864, 0.867, 0.871,
                               0.874, 0.876, 0.879, 0.881, 0.883, 0.885, 0.886, 0.888, 0.889, 0.891]),
    'F1_Macro_Media': np.array([0.004, 0.008, 0.012, 0.015, 0.018, 0.021, 0.024, 0.028, 0.031, 0.034,
                               0.062, 0.084, 0.103, 0.119, 0.135, 0.147, 0.158, 0.168, 0.178,
                               0.251, 0.295, 0.328, 0.350, 0.373, 0.393, 0.406, 0.420, 0.433,
                               0.503, 0.549, 0.574, 0.595, 0.612, 0.625, 0.635, 0.645, 0.654,
                               0.661, 0.670, 0.675, 0.679, 0.685, 0.689, 0.693, 0.698, 0.701, 0.704])
}
df_performance_media = pd.DataFrame(data_l0_performance)

x_data = df_performance_media['Tam_L0'].values
y_acuracia_media = df_performance_media['Acuracia_Media'].values
y_f1_media = df_performance_media['F1_Macro_Media'].values

In [ ]:
# --- Função Exponencial Saturante ---
# y = a * (1 - exp(-b * x)) + c
# a: Amplitude do crescimento (performance_max - performance_inicial)
# b: Taxa de crescimento
# c: Performance inicial (quando x é muito pequeno, ou o valor em x=0 se o modelo permitir)
def func_exp_saturante(x, a, b, c):
    return a * (1 - np.exp(-b * x)) + c

# --- Função Logarítmica ---
# y = a * ln(x) + b
# Adicionar um pequeno epsilon a x para evitar log(0) se x_data puder ser 0
def func_logaritmica(x, a, b):
    return a * np.log(x + 1e-9) + b # Adicionado 1e-9 para estabilidade numérica

# --- Função de Potência ---
# y = a * x^b
def func_potencia(x, a, b):
    return a * np.power(x + 1e-9, b)


# --- Ajuste para Acurácia ---
print("Ajustando curva para Acurácia Média...")
try:
    # Estimativas iniciais (p0) podem ajudar na convergência
    # Para func_exp_saturante: [amplitude_aprox, taxa_aprox, y_inicial_aprox]
    # p0_acc = [np.max(y_acuracia_media) - np.min(y_acuracia_media), 0.0001, np.min(y_acuracia_media)]
    # params_acc, covariance_acc = curve_fit(func_exp_saturante, x_data, y_acuracia_media, p0=p0_acc, maxfev=5000)
    # a_acc, b_acc, c_acc = params_acc
    # print(f"Parâmetros ajustados para Acurácia (exp saturante): a={a_acc:.4f}, b={b_acc:.6f}, c={c_acc:.4f}")
    # y_pred_acc = func_exp_saturante(x_data, a_acc, b_acc, c_acc)

    # Alternativa: Logarítmica
    p0_log_acc = [0.1, 0.1] # [a_aprox, b_aprox]
    params_log_acc, _ = curve_fit(func_logaritmica, x_data, y_acuracia_media, p0=p0_log_acc, maxfev=5000)
    print(f"Parâmetros ajustados para Acurácia (logarítmica): a={params_log_acc[0]:.4f}, b={params_log_acc[1]:.4f}")
    y_pred_acc_log = func_logaritmica(x_data, *params_log_acc)

except RuntimeError:
    print("Não foi possível ajustar a curva exponencial saturante para Acurácia. Tente outras estimativas iniciais ou outra função.")
    y_pred_acc = None # Para evitar erro no plot

# --- Ajuste para F1-Macro ---
print("\nAjustando curva para F1-Macro Médio...")
try:
    p0_f1 = [np.max(y_f1_media) - np.min(y_f1_media), 0.0001, np.min(y_f1_media)]
    params_f1, covariance_f1 = curve_fit(func_exp_saturante, x_data, y_f1_media, p0=p0_f1, maxfev=5000)
    a_f1, b_f1, c_f1 = params_f1
    print(f"Parâmetros ajustados para F1-Macro (exp saturante): a={a_f1:.4f}, b={b_f1:.6f}, c={c_f1:.4f}")
    y_pred_f1 = func_exp_saturante(x_data, a_f1, b_f1, c_f1)
except RuntimeError:
    print("Não foi possível ajustar a curva exponencial saturante para F1-Macro.")
    y_pred_f1 = None

# --- Plotagem ---
plt.style.use('seaborn-v0_8-whitegrid') # Estilo
fig, ax = plt.subplots(figsize=(12, 7))

# Plot dos dados médios originais
ax.plot(x_data, y_acuracia_media, 'o', color='crimson', markersize=5, label='Acurácia Média (Dados)')
if y_pred_acc is not None:
    ax.plot(x_data, y_pred_acc, '-', color='lightcoral', linewidth=2, label=f'Acurácia Ajustada (Exp Sat.)\n  $y={a_acc:.2f}(1-e^{{-{b_acc:.5f}x}}) + {c_acc:.2f}$')

ax.plot(x_data, y_f1_media, 's', color='darkcyan', markersize=5, label='F1-Macro Médio (Dados)')
if y_pred_f1 is not None:
    ax.plot(x_data, y_pred_f1, '--', color='mediumturquoise', linewidth=2, label=f'F1-Macro Ajustado (Exp Sat.)\n  $y={a_f1:.2f}(1-e^{{-{b_f1:.5f}x}}) + {c_f1:.2f}$')

# Configurações do Gráfico
ax.set_xlabel('Tamanho da Amostra Inicial (L0)')
ax.set_ylabel('Performance Média')
ax.set_title('Regressão das Curvas de Performance Média por Tamanho de L0')
ax.legend(loc='center right', bbox_to_anchor=(1.0, 0.4))
ax.set_xscale('log') # Usar escala logarítmica para o eixo X para melhor visualização
ax.grid(True, which="both", ls="--", c='0.7')
plt.tight_layout()
plt.show()

In [ ]:
def r_squared(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred)**2) # Soma dos quadrados dos resíduos
    ss_tot = np.sum((y_true - np.mean(y_true))**2) # Soma total dos quadrados
    if ss_tot == 0: # Evita divisão por zero se y_true for constante
        return 1.0 if ss_res == 0 else 0.0
    return 1 - (ss_res / ss_tot)

if y_pred_acc is not None:
    r2_acc = r_squared(y_acuracia_media, y_pred_acc)
    print(f"\nR-quadrado para Acurácia (Exp Saturante): {r2_acc:.4f}")

if y_pred_f1 is not None:
    r2_f1 = r_squared(y_f1_media, y_pred_f1)
    print(f"R-quadrado para F1-Macro (Exp Saturante): {r2_f1:.4f}")

In [ ]:
from sklearn.metrics import mean_squared_error

if y_pred_acc is not None:
    mse_acc = mean_squared_error(y_acuracia_media, y_pred_acc)
    rmse_acc = np.sqrt(mse_acc)
    print(f"RMSE para Acurácia (Exp Saturante): {rmse_acc:.4f}")

if y_pred_f1 is not None:
    mse_f1 = mean_squared_error(y_f1_media, y_pred_f1)
    rmse_f1 = np.sqrt(mse_f1)
    print(f"RMSE para F1-Macro (Exp Saturante): {rmse_f1:.4f}")

In [ ]:
if y_pred_acc is not None:
    residuos_acc = y_acuracia_media - y_pred_acc
    plt.figure(figsize=(10, 4))
    plt.scatter(y_pred_acc, residuos_acc, alpha=0.5)
    plt.axhline(0, color='red', linestyle='--')
    plt.xlabel('Valores Preditos (Acurácia)')
    plt.ylabel('Resíduos')
    plt.title('Plot de Resíduos para Acurácia (Exp Saturante)')
    plt.show()

In [ ]:
from scipy.stats import shapiro
if y_pred_acc is not None and len(residuos_acc) >=3 : # Shapiro-Wilk requer pelo menos 3 amostras
    stat, p_shapiro = shapiro(residuos_acc)
    print(f'\nTeste de Shapiro-Wilk para resíduos da Acurácia: Estatística={stat:.3f}, p-valor={p_shapiro:.3f}')
    if p_shapiro > 0.05:
        print('Resíduos parecem normalmente distribuídos (não rejeita H0)')
    else:
        print('Resíduos não parecem normalmente distribuídos (rejeita H0)')